# VectorComponents — vector (multi-component) unknown

`MagneticStandalone` · ASN 28 validation case (TRUST-LiMe).


## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe: $\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$,
with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``vector_components``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

`nb_composantes` is omitted, so the default (one component per space direction)
is used (i.e. 2 in 2D). On the unit square with Dirichlet $b=0$ on all faces,
the Laplacian eigenmodes $\sin(m\pi x)\sin(n\pi y)$ decay at rate
$\eta(m^2+n^2)\pi^2$. The two components are given **different** modes so that
a component swap or a mixed assembly changes the rate and is caught:

$$b_x=\sin(\pi x)\sin(\pi y)e^{-2\eta\pi^2 t},\quad b_y=\sin(2\pi x)\sin(\pi y)e^{-5\eta\pi^2 t}.$$

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 4 | components transported independently, no coupling | two distinct eigen-rates |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Sequential-only: the dataset is run by `trustutils` on one processor (no `partition()`, no `nbProcs`).


In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "vector_components.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("vector_components_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
import matplotlib.pyplot as plt

ETA = 0.02
def bx_ratio(t): return np.exp(-2 * ETA * np.pi**2 * np.asarray(t, float))
def by_ratio(t): return np.exp(-5 * ETA * np.pi**2 * np.asarray(t, float))

err_bx = ratio_error("vector_components", ["SONDE_BX"], bx_ratio)
err_by = ratio_error("vector_components", ["SONDE_BY"], by_ratio)
print(f"component 0 (rate 2 eta pi^2) max error : {err_bx:.4e}")
print(f"component 1 (rate 5 eta pi^2) max error : {err_by:.4e}")

# a component swap would make the two rates coincide / mismatch strongly
assert err_bx < 0.03, f"component 0 error too large: {err_bx:.3e}"
assert err_by < 0.03, f"component 1 error too large: {err_by:.3e}"
# sanity: the two rates are genuinely different (guards the test itself)
assert abs(np.exp(-2*ETA*np.pi**2*0.5) - np.exp(-5*ETA*np.pi**2*0.5)) > 0.2
print("OK: components decay independently at their own eigen-rates.")

plt.figure(figsize=(6,4))
t, bx = read_son("vector_components", "SONDE_BX")
t, by = read_son("vector_components", "SONDE_BY")
plt.plot(t, bx/bx[0], label="bx (2 eta pi^2)")
plt.plot(t, by/by[0], label="by (5 eta pi^2)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()


## Computer Performance


In [ ]:
run.tablePerf()